# ASL LSTM Sign Recognizer — Training Notebook (Google Colab)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU` for faster training.

This notebook trains an LSTM model to recognise ASL signs from MediaPipe hand landmarks.
All intermediate files are saved to **Google Drive** so you never lose progress if the session disconnects.

Run all cells top-to-bottom in a fresh Colab session.

## ⚙️ Configuration — edit this cell only

Set your Google Drive folder path and whether to use demo (synthetic) data or the real dataset.

In [ ]:
# ── Change these to match your Google Drive layout ──────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/asl_translator'  # folder in your Drive

# Set to False when you have the real ASL Citizen dataset ready
DEMO_MODE    = True

# ── Constants (no need to change) ───────────────────────────────────────────
MAX_FRAMES   = 30   # frames per clip fed to the LSTM
N_COORDS     = 63   # 21 landmarks × 3 (x, y, z)
BATCH_SIZE   = 32
MAX_EPOCHS   = 100

---
## Section 1: Dataset Setup

Mount Google Drive, install dependencies, download the ASL Citizen dataset,
extract MediaPipe Hands landmarks from video frames, and save `.npy` arrays to Drive.

In [ ]:
# Mount Google Drive — you will be prompted to authorise access
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_DIR = Path(DRIVE_BASE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

In [ ]:
# Install dependencies (takes ~2 min on first run)
!pip install -q mediapipe tensorflow scikit-learn matplotlib seaborn datasets tqdm
print('Dependencies installed.')

### 1a. Load the ASL Citizen Dataset

The ASL Citizen dataset contains ~83 000 short MP4 clips of ASL signs.
Load it from HuggingFace — no account needed, just internet access.

```python
from datasets import load_dataset
ds = load_dataset('google/asl_citizen', split='train')
```

The cell below does this automatically when `DEMO_MODE = False`.
In `DEMO_MODE = True` it generates synthetic data so you can test the full pipeline instantly.

In [ ]:
import numpy as np

if DEMO_MODE:
    print('DEMO MODE — generating synthetic data (no real videos needed).')
    print('Set DEMO_MODE = False in the config cell to train on real data.\n')
    LETTERS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
    WORDS   = ['hello', 'thank_you', 'yes', 'no', 'please', 'sorry',
               'help', 'more', 'stop', 'go', 'eat', 'drink', 'home',
               'work', 'good', 'bad', 'love', 'friend', 'family', 'name']
    ALL_LABELS = LETTERS + WORDS
    rng = np.random.default_rng(42)
    X_list, y_list = [], []
    for label in ALL_LABELS:
        for _ in range(50):  # 50 samples per class
            X_list.append(rng.random((MAX_FRAMES, N_COORDS), dtype=np.float32))
            y_list.append(label)
    X     = np.stack(X_list)
    y_raw = np.array(y_list)
    print(f'Synthetic data — X: {X.shape}, classes: {len(ALL_LABELS)}')
else:
    # Real dataset — loaded in Section 1b below
    from datasets import load_dataset
    print('Loading ASL Citizen from HuggingFace (this may take a few minutes)...')
    hf_ds = load_dataset('google/asl_citizen', split='train')
    print(hf_ds)

### 1b. Extract MediaPipe Landmarks (real data only)

For each video clip:
1. Decode frames with OpenCV
2. Run MediaPipe Hands on each frame
3. Flatten 21 landmarks × 3 coords → 63 floats
4. Pad / truncate to `MAX_FRAMES`

**This is the slow step (~3–6 hours for the full dataset on Colab free tier).**
Progress is saved to Drive every 500 clips so you can resume if the session disconnects.

In [ ]:
if not DEMO_MODE:
    import cv2
    import mediapipe as mp
    from tqdm import tqdm

    mp_hands = mp.solutions.hands
    CHECKPOINT_NPY = OUTPUT_DIR / 'extraction_checkpoint.npz'
    SAVE_EVERY     = 500  # save progress to Drive every N clips

    def extract_landmarks_from_video(video_path, max_frames=MAX_FRAMES):
        """Return ndarray (max_frames, 63). Zero-fills frames with no hand."""
        frames = []
        cap = cv2.VideoCapture(str(video_path))
        with mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                            min_detection_confidence=0.5,
                            min_tracking_confidence=0.5) as hands:
            while cap.isOpened() and len(frames) < max_frames:
                ret, frame = cap.read()
                if not ret:
                    break
                result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if result.multi_hand_landmarks:
                    lm = result.multi_hand_landmarks[0]
                    coords = [v for lmk in lm.landmark for v in (lmk.x, lmk.y, lmk.z)]
                    frames.append(coords)
                else:
                    frames.append([0.0] * N_COORDS)
        cap.release()
        pad_len = max_frames - len(frames)
        if pad_len > 0:
            frames += [[0.0] * N_COORDS] * pad_len
        return np.array(frames[:max_frames], dtype=np.float32)

    # Resume from checkpoint if it exists
    if CHECKPOINT_NPY.exists():
        ckpt   = np.load(CHECKPOINT_NPY, allow_pickle=True)
        X_list = list(ckpt['X'])
        y_list = list(ckpt['y'])
        start  = len(X_list)
        print(f'Resuming from checkpoint at clip {start}')
    else:
        X_list, y_list, start = [], [], 0

    for i, row in enumerate(tqdm(hf_ds, desc='Extracting landmarks', initial=start)):
        if i < start:
            continue
        try:
            lm = extract_landmarks_from_video(row['video']['path'])
            X_list.append(lm)
            y_list.append(row['label'])
        except Exception as e:
            print(f'  Skipping clip {i}: {e}')
        # Save checkpoint to Drive every SAVE_EVERY clips
        if (i + 1) % SAVE_EVERY == 0:
            np.savez(CHECKPOINT_NPY, X=np.stack(X_list), y=np.array(y_list))
            print(f'  Checkpoint saved at clip {i + 1}')

    X     = np.stack(X_list)
    y_raw = np.array(y_list)
    print(f'Extraction complete — X: {X.shape}')

In [ ]:
# Save landmark arrays to Drive
np.save(OUTPUT_DIR / 'X.npy',     X)
np.save(OUTPUT_DIR / 'y_raw.npy', y_raw)
print(f'Saved X.npy {X.shape} and y_raw.npy {y_raw.shape} to Drive')

---
## Section 2: Data Preprocessing

Load arrays from Drive, z-score normalise landmarks, encode labels, split 80/20.

In [ ]:
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(DRIVE_BASE)
X     = np.load(OUTPUT_DIR / 'X.npy')
y_raw = np.load(OUTPUT_DIR / 'y_raw.npy', allow_pickle=True)
print(f'Loaded — X: {X.shape}, unique labels: {len(np.unique(y_raw))}')

In [ ]:
# Z-score normalise each frame independently (makes model scale/position invariant)
mean  = X.mean(axis=-1, keepdims=True)
std   = X.std(axis=-1,  keepdims=True) + 1e-8
X_norm = (X - mean) / std
print(f'Normalised — mean: {X_norm.mean():.4f}, std: {X_norm.std():.4f}')

In [ ]:
import json
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le          = LabelEncoder()
y_int       = le.fit_transform(y_raw)
num_classes = len(le.classes_)
y_cat       = to_categorical(y_int, num_classes)

# Save label_encoder.json — needed by the backend at inference time
label_map = {int(i): str(c) for i, c in enumerate(le.classes_)}
with open(OUTPUT_DIR / 'label_encoder.json', 'w') as f:
    json.dump(label_map, f, indent=2)

print(f'num_classes: {num_classes}')
print(f'Saved label_encoder.json to Drive')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y_cat, test_size=0.2, random_state=42, stratify=y_int
)
print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')

---
## Section 3: LSTM Model Definition

`Input(30, 63) → LSTM(128) → Dropout → LSTM(64) → Dropout → Dense(num_classes, softmax)`

Kept small so it loads within 5 seconds on standard hardware.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

print(f'TensorFlow {tf.__version__}  |  GPU: {tf.config.list_physical_devices("GPU")}')

model = Sequential([
    Input(shape=(MAX_FRAMES, N_COORDS)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(num_classes, activation='softmax'),
], name='asl_lstm')

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

---
## Section 4: Training

Checkpoints are saved to Drive after every epoch so training can be resumed.
EarlyStopping stops training when validation accuracy stops improving.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

BEST_WEIGHTS = str(OUTPUT_DIR / 'best_weights.h5')

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=BEST_WEIGHTS, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)
print('Training complete. Best weights saved to Drive.')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(history.history[metric],          label=f'Train {metric}')
    ax.plot(history.history[f'val_{metric}'], label=f'Val {metric}')
    ax.set_title(metric.capitalize())
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png to Drive')

---
## Section 5: Evaluation

Confusion matrix, per-class accuracy, and the 5 weakest signs.

In [ ]:
import numpy as np

model.load_weights(BEST_WEIGHTS)
y_pred_int = np.argmax(model.predict(X_val, verbose=0), axis=1)
y_true_int = np.argmax(y_val, axis=1)
print(f'Overall validation accuracy: {np.mean(y_pred_int == y_true_int):.4f}')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm          = confusion_matrix(y_true_int, y_pred_int)
class_names = le.classes_
sz          = max(12, num_classes // 2)

fig, ax = plt.subplots(figsize=(sz, sz))
sns.heatmap(cm, annot=(num_classes <= 30), fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png to Drive')

In [ ]:
per_class_acc = {}
for idx, name in enumerate(class_names):
    mask = y_true_int == idx
    per_class_acc[name] = float(np.mean(y_pred_int[mask] == idx)) if mask.sum() else float('nan')

print('Per-class accuracy:')
for name, acc in sorted(per_class_acc.items(), key=lambda x: x[1] if not np.isnan(x[1]) else -1):
    bar = '#' * int(acc * 20) if not np.isnan(acc) else '(no samples)'
    print(f'  {name:<20s} {acc:.3f}  {bar}')

weakest = sorted(((k, v) for k, v in per_class_acc.items() if not np.isnan(v)), key=lambda x: x[1])[:5]
print('\n5 weakest signs:')
for i, (name, acc) in enumerate(weakest, 1):
    print(f'  {i}. {name:<20s}  {acc:.3f}')

---
## Section 6: Export

Save `sign_recognizer.h5` and `label_encoder.json` to Drive, then download them
to your local machine. Copy both files into the `models/` directory of the backend.

In [ ]:
H5_PATH = OUTPUT_DIR / 'sign_recognizer.h5'
model.save(str(H5_PATH))
print(f'Saved {H5_PATH}')

In [ ]:
# Optional: TFLite conversion (smaller file, faster inference on CPU)
import tensorflow as tf

TFLITE_PATH = OUTPUT_DIR / 'sign_recognizer.tflite'
converter   = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
TFLITE_PATH.write_bytes(tflite_bytes)
print(f'Saved {TFLITE_PATH}  ({len(tflite_bytes) / 1024:.1f} KB)')

In [ ]:
# Download artifacts to your local machine
from google.colab import files

artifacts = [
    OUTPUT_DIR / 'sign_recognizer.h5',
    OUTPUT_DIR / 'sign_recognizer.tflite',
    OUTPUT_DIR / 'label_encoder.json',
    OUTPUT_DIR / 'training_curves.png',
    OUTPUT_DIR / 'confusion_matrix.png',
]

for path in artifacts:
    if Path(path).exists():
        files.download(str(path))
        print(f'Downloading {Path(path).name}...')

In [ ]:
print('=== Exported Artifacts ===')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:<40s}  {f.stat().st_size / 1024:8.1f} KB')

print('\nNext steps:')
print('  1. Copy sign_recognizer.h5  → models/sign_recognizer.h5')
print('  2. Copy label_encoder.json  → models/label_encoder.json')
print('  3. Start the FastAPI backend — it will load the model automatically.')